# Utilities

In [1]:
import scipy.stats as st
from scipy.stats import binom
import xarray as xr
from scipy.stats import linregress
from EWS_functions import *
from EWS_wrappers import *
import pandas as pd
import numpy as np


In [2]:
# DATA PATHS
# ----------
# Set these to the directories where your large data files are stored.
# These files are too large to include in the repository.
# EWS_DIR  : directory containing the computed EWS arrays (lambdas, AR1s, stds)
#             for all CMIP6 models. Relative path from this notebook's location.
EWS_DIR = '../../new_AR6'


In [3]:
rss = ['r{}'.format(i) for i in np.arange(1,11)]


In [4]:
labels = ['(a)','(b)','(c)','(d)','(e)','(f)','(g)','(h)','(i)','(j)','(k)','(l)','(m)','(n)','(o)','(p)']


In [5]:
# function to get the percentile of the last value in an array, ignoring nans
# applies 
def f(x):
    if not np.isnan(x[-1]):
        x=x[~np.isnan(x)]
        return st.percentileofscore(x, x[-1])
    else:
        return np.nan


In [6]:
def get_corr(x,y):
    y = y[~np.isnan(x)]
    x = x[~np.isnan(x)]
    slope, intercept, r, p, se = linregress(x,y)
    return r, p


In [7]:
def get_pvs(true_trends,surr_trends,dim='surrogates'):
    d = true_trends.expand_dims(dim=dim, axis=0)
    d[dim]=[int(len(surr_trends[dim].values))]
    parr = xr.concat((surr_trends,d),dim=dim) # the last value on the row is the true trend
    ppvs = 1- np.apply_along_axis(f, 3, parr)/100
    return ppvs


In [8]:
def make_string(idc):
    string=''
    for i in idc:
        string = string +'r{} '.format(i+1)
    return string


# data

## AMOC timeseries

In [ ]:
amoc = xr.open_dataset('../data/CMIP6_amoc2.nc').amoc
strn26 = amoc.sel(indices='strn26')
strn35 = amoc.sel(indices='strn35')
index = amoc.sel(indices='index')
yrs = strn26.time.values
models = strn26.models.values
ensembs = strn26.ensemble_members.values


In [10]:
model_names = []
for mod in models:
    n_tot = np.count_nonzero(~np.isnan(strn35.sel(models=mod).values))
    if n_tot!=0:
        model_names.append(mod)


## AMOC trends

In [11]:
strn26_trend = np.zeros((34,10))
for imod in range(34):
    for iens in range(10):
        amc = np.nan_to_num(strn26.values[imod,iens,:])
        slope, intercept, r, p, se = linregress(yrs, amc)
        strn26_trend[imod,iens]=slope*100 # trend of Sv/century
index_trend = np.zeros((34,10))
for imod in range(34):
    for iens in range(10):
        idx = np.nan_to_num(index.values[imod,iens,:])
        slope, intercept, r, p, se = linregress(yrs, idx)
        index_trend[imod,iens]=slope*100 # trend of Sv/century
strn35_trend = np.zeros((34,10))
for imod in range(34):
    for iens in range(10):
        amc = np.nan_to_num(strn35.values[imod,iens,:])
        slope, intercept, r, p, se = linregress(yrs, amc)
        strn35_trend[imod,iens]=slope*100 # trend of Sv/century


## EWS p-values

### ws=60 (main text)

In [12]:
cmip_trends = xr.open_dataset(f'{EWS_DIR}/CMIP6_lambdas_w60_rmean_trends.nc').polyfit_coefficients.sel(degree=1)


In [ ]:
all_EWS = xr.open_dataset('../data/CMIP6_all_EWS_ppvs.nc')
ar1_ppvs = all_EWS.ppvs.sel(indicators='ar1')
std_ppvs = all_EWS.ppvs.sel(indicators='var')
ppvs = all_EWS.ppvs.sel(indicators='lambda')


### ws=50

In [14]:
cmip_trends50 = xr.open_dataset(f'{EWS_DIR}/CMIP6_stds_w50_rmean_trends.nc').polyfit_coefficients.sel(degree=1)


In [ ]:
all_EWS50 = xr.open_dataset('../data/CMIP6_all_EWS_ppvs_w50.nc')
ar1_ppvs50 = all_EWS50.ppvs.sel(indicators='ar1')
std_ppvs50 = all_EWS50.ppvs.sel(indicators='var')
ppvs50 = all_EWS50.ppvs.sel(indicators='lambda')


### ws=70

In [16]:
cmip_trends70 = xr.open_dataset(f'{EWS_DIR}/CMIP6_stds_w70_rmean_trends.nc').polyfit_coefficients.sel(degree=1)


In [ ]:
all_EWS70 = xr.open_dataset('../data/CMIP6_all_EWS_ppvs_w70.nc')
ar1_ppvs70 = all_EWS70.ppvs.sel(indicators='ar1')
std_ppvs70 = all_EWS70.ppvs.sel(indicators='var')
ppvs70 = all_EWS70.ppvs.sel(indicators='lambda')


# Counts for slice calculations

In [18]:
counts = np.full(ppvs.shape,np.nan)
counts[np.where((ar1_ppvs<=0.05)&(std_ppvs<=0.05)&(ppvs<=0.05))]=3
counts[np.where(
    ((ar1_ppvs<=0.05)&(std_ppvs<=0.05)&(ppvs>0.05)) |
   ((ar1_ppvs<=0.05)&(std_ppvs>0.05)&(ppvs<=0.05)) |
    ((ar1_ppvs>0.05)&(std_ppvs<=0.05)&(ppvs<=0.05)))       
      ]=2
counts[np.where(
    ((ar1_ppvs<=0.05)&(std_ppvs>0.05)&(ppvs>0.05)) |
   ((ar1_ppvs>0.05)&(std_ppvs>0.05)&(ppvs<=0.05)) |
    ((ar1_ppvs>0.05)&(std_ppvs<=0.05)&(ppvs>0.05)))       
      ]=1


In [19]:
counts50 = np.full(ppvs50.shape,np.nan)
counts50[np.where((ar1_ppvs50<=0.05)&(std_ppvs50<=0.05)&(ppvs50<=0.05))]=3
counts50[np.where(
    ((ar1_ppvs50<=0.05)&(std_ppvs50<=0.05)&(ppvs50>0.05)) |
   ((ar1_ppvs50<=0.05)&(std_ppvs50>0.05)&(ppvs50<=0.05)) |
    ((ar1_ppvs50>0.05)&(std_ppvs50<=0.05)&(ppvs50<=0.05)))       
      ]=2
counts50[np.where(
    ((ar1_ppvs50<=0.05)&(std_ppvs50>0.05)&(ppvs50>0.05)) |
   ((ar1_ppvs50>0.05)&(std_ppvs50>0.05)&(ppvs50<=0.05)) |
    ((ar1_ppvs50>0.05)&(std_ppvs50<=0.05)&(ppvs50>0.05)))       
      ]=1


In [20]:
counts70 = np.full(ppvs70.shape,np.nan)
counts70[np.where((ar1_ppvs70<=0.05)&(std_ppvs70<=0.05)&(ppvs70<=0.05))]=3
counts70[np.where(
    ((ar1_ppvs70<=0.05)&(std_ppvs70<=0.05)&(ppvs70>0.05)) |
   ((ar1_ppvs70<=0.05)&(std_ppvs70>0.05)&(ppvs70<=0.05)) |
    ((ar1_ppvs70>0.05)&(std_ppvs70<=0.05)&(ppvs70<=0.05)))       
      ]=2
counts70[np.where(
    ((ar1_ppvs70<=0.05)&(std_ppvs70>0.05)&(ppvs70>0.05)) |
   ((ar1_ppvs70>0.05)&(std_ppvs70>0.05)&(ppvs70<=0.05)) |
    ((ar1_ppvs70>0.05)&(std_ppvs70<=0.05)&(ppvs70>0.05)))       
      ]=1


# Table A in S2 Appendix

In [21]:
array = np.zeros((4,4))
for i in range(3):
    array[0,i]=len(np.where(~np.isnan(counts[i]))[0])
    array[1,i]=len(np.where(counts[i]==1)[0])
    array[2,i]=len(np.where(counts[i]==2)[0])
    array[3,i]=len(np.where(counts[i]==3)[0])

array[0,3]=len(np.where(~np.isnan(counts))[0])
array[1,3]=len(np.where(counts==1)[0])
array[2,3]=len(np.where(counts==2)[0])
array[3,3]=len(np.where(counts==3)[0])    


In [ ]:
df = pd.DataFrame(data=array,index=['Any','One','Two','Three'],columns=['S26','S35','SSTI','Any'])
df.to_csv('../data/EWS_numbers.txt')
df


,S26,S35,SSTI,Any
Any,28.0,32.0,13.0,73.0
One,17.0,21.0,9.0,47.0
Two,10.0,6.0,4.0,20.0
Three,1.0,5.0,0.0,6.0


# Table B in S2 Appendix

In [23]:
probabilities = np.full((3,3,34),np.nan)
min_p = 0.5/27
for j, indc in enumerate(['lambda','ar1','var']):
    for im, model in enumerate(models):
        dd = all_EWS.ppvs.sel(models=model).sel(indicators=indc)
        if np.sum(dd)!=0:
            for i, idx in enumerate(['strn26','strn35','index']):
                nens = np.count_nonzero(~np.isnan(cmip_trends.sel(indices='strn26').sel(models=model).values))
                n_tot= nens
                ews_tot = len(np.where(dd.sel(indices=idx)<=0.05)[0])
                prob = binom(n_tot,0.05).pmf(ews_tot) # the probability of getting exactly the number of <=0.05 in the specific model, index and strength
                if (1-binom(27,prob).cdf(0))<=0.05:
                    print(model, indc, idx,'{:.4f}'.format((1-binom(27,prob).cdf(0)))) # equation 2 from S2 Appendix
                    # the probability that an event with probability prob happens at least once in 27 models
                probabilities[j,i,im]=prob


CanESM5 lambda strn35 0.0257
CESM2 lambda strn35 0.0257
MIROC6 ar1 strn35 0.0257
HadGEM3-GC31-LL ar1 strn35 0.0127
CESM2 ar1 strn26 0.0257


# Table C in S2 Appendix

In [24]:
# table C in S2 appendix 

probabilities = np.full((3,3),np.nan)
min_p = 0.5/27
for j, indc in enumerate(['lambda','ar1','var']):
    for i, idx in enumerate(['strn26','strn35','index']):
        dd = all_EWS.ppvs.sel(indicators=indc).sel(indices=idx)
        ews_tot = len(np.where(dd<=0.05)[0])
        prob = binom(133,0.05).pmf(ews_tot)
        probabilities[j,i]='{:.2e}'.format(prob)

probabilities


array([[4.17e-03, 8.31e-05, 1.04e-01],
       [2.45e-04, 6.76e-04, 1.52e-01],
       [9.19e-02, 4.17e-03, 1.59e-01]])

# Table A in S4 Appendix

In [25]:
lmodels = []
lens = []
for im, model in enumerate(models):
    idc = np.where(~np.isnan(amoc.sel(indices='index').sel(models=model).isel(time=0).values))
    n_tot = np.count_nonzero(~np.isnan(amoc.sel(indices='index').sel(models=model).isel(time=0).values))
    idc = np.where(~np.isnan(amoc.sel(indices='index').sel(models=model).isel(time=0).values))
    if n_tot!=0:
        lmodels.append(model)
        lens.append(idc[0])


In [26]:
lis = []
for im, model in enumerate(lmodels):
    lis.append([model,make_string(lens[im])])


In [ ]:
df = pd.DataFrame(data=lis,index=None)
df.to_csv('../data/data_list.txt',index=False)
df


,0,1
0,BCC-CSM2-MR,r1 r2 r3
1,BCC-ESM1,r1 r2 r3
2,CAMS-CSM1-0,r1 r2
3,CanESM5,r1 r2 r3 r4 r5 r6 r7 r8 r9 r10
4,CNRM-CM6-1,r1 r2 r3 r4 r5 r6 r7 r8 r9 r10
5,CNRM-CM6-1-HR,r1
6,CNRM-ESM2-1,r1 r2 r3 r4 r5
7,E3SM-1-1,r1
8,EC-Earth3,r1 r2 r4 r7
9,EC-Earth3-Veg,r1 r2 r3 r4


# Table D in S4 Appendix

In [ ]:
namess = ['S26.5','S35','SSTI']

liss = []
ir = 0
for im, model in enumerate(models):
    idc = np.where(~np.isnan(cmip_trends.sel(indices='strn26').sel(models=model).values))
    n_tot = np.count_nonzero(~np.isnan(cmip_trends.sel(indices='strn26').sel(models=model).values))
    inliss = []
    if n_tot!=0:
        for idx in np.arange(0,10):
            string = ''
            for ifng in range(3):
                data = ppvs[ifng][im][idx]
                if data<=0.05:
                    string = string + namess[ifng] +' '
            inliss.append(string)
        liss.append(inliss)
df = pd.DataFrame(data=liss,index=lmodels,columns=rss)
df.to_csv('../data/lam_list.txt',sep=',')


# Table E in S4 Appendix

In [ ]:
namess = ['S26.5','S35','SSTI']

liss = []
ir = 0
for im, model in enumerate(models):
    idc = np.where(~np.isnan(cmip_trends.sel(indices='strn26').sel(models=model).values))
    n_tot = np.count_nonzero(~np.isnan(cmip_trends.sel(indices='strn26').sel(models=model).values))
    inliss = []
    if n_tot!=0:
        for idx in np.arange(0,10):
            string = ''
            for ifng in range(3):
                data = ar1_ppvs[ifng][im][idx]
                if data<=0.05:
                    string = string + namess[ifng] +' '
            inliss.append(string)
        liss.append(inliss)
df = pd.DataFrame(data=liss,index=lmodels,columns=rss)
df.to_csv('../data/ar1_list.txt',sep=',')


# Table F in S4 Appendix

In [ ]:
namess = ['S26.5','S35','SSTI']

liss = []
ir = 0
for im, model in enumerate(models):
    idc = np.where(~np.isnan(cmip_trends.sel(indices='strn26').sel(models=model).values))
    n_tot = np.count_nonzero(~np.isnan(cmip_trends.sel(indices='strn26').sel(models=model).values))
    inliss = []
    if n_tot!=0:
        for idx in np.arange(0,10):
            string = ''
            for ifng in range(3):
                data = std_ppvs[ifng][im][idx]
                if data<=0.05:
                    string = string + namess[ifng] +' '
            inliss.append(string)
        liss.append(inliss)
df = pd.DataFrame(data=liss,index=lmodels,columns=rss)
df.to_csv('../data/std_list.txt',sep=',')


# Table B in S4 Appendix

In [ ]:
array = np.zeros((4,4))
for i in range(3):
    array[0,i]=len(np.where(~np.isnan(counts50[i]))[0])
    array[1,i]=len(np.where(counts50[i]==1)[0])
    array[2,i]=len(np.where(counts50[i]==2)[0])
    array[3,i]=len(np.where(counts50[i]==3)[0])

array[0,3]=len(np.where(~np.isnan(counts50))[0])
array[1,3]=len(np.where(counts50==1)[0])
array[2,3]=len(np.where(counts50==2)[0])
array[3,3]=len(np.where(counts50==3)[0])    

df = pd.DataFrame(data=array,index=['Any','One','Two','Three'],columns=['S26','S35','SSTI','Any'])
df.to_csv('../data/EWS_numbers_w50.txt')
df


,S26,S35,SSTI,Any
Any,24.0,35.0,21.0,80.0
One,15.0,21.0,18.0,54.0
Two,7.0,11.0,3.0,21.0
Three,2.0,3.0,0.0,5.0


# Table C in S4 Appendix

In [ ]:
array = np.zeros((4,4))
for i in range(3):
    array[0,i]=len(np.where(~np.isnan(counts70[i]))[0])
    array[1,i]=len(np.where(counts70[i]==1)[0])
    array[2,i]=len(np.where(counts70[i]==2)[0])
    array[3,i]=len(np.where(counts70[i]==3)[0])

array[0,3]=len(np.where(~np.isnan(counts70))[0])
array[1,3]=len(np.where(counts70==1)[0])
array[2,3]=len(np.where(counts70==2)[0])
array[3,3]=len(np.where(counts70==3)[0])    

df = pd.DataFrame(data=array,index=['Any','One','Two','Three'],columns=['S26','S35','SSTI','Any'])
df.to_csv('../data/EWS_numbers_w70.txt')
df


,S26,S35,SSTI,Any
Any,31.0,40.0,14.0,85.0
One,17.0,23.0,7.0,47.0
Two,11.0,10.0,6.0,27.0
Three,3.0,7.0,1.0,11.0
